# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
!git clone https://github.com/Ahmedali3ff/Flyrank-internship-.git

Cloning into 'Flyrank-internship-'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 160 (delta 64), reused 99 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 1.88 MiB | 16.73 MiB/s, done.
Resolving deltas: 100% (64/64), done.


In [9]:
import pandas as pd
from pathlib import Path

# Find the dataset inside the cloned repository
matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError(
            "Dataset not found. Make sure the FlyRank repository is available in Colab."
                )

data_path = matches[0]
print("Dataset:", data_path)

                # Load data
df = pd.read_csv(data_path)

                # Create the target label
df["is_declining_label"] = (
df["trend_direction"].astype(str).str.lower() == "down"
                    ).astype(int)

                    # Basic checks
print("\nDataset shape:", df.shape)
print("Declining rate:", round(df["is_declining_label"].mean(), 4))

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts().sort_index())

Dataset: /content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv

Dataset shape: (30000, 45)
Declining rate: 0.5421

Target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64


In [11]:
# Features used by the model
feature_cols = [
    "search_volume",
        "competition",
            "cpc",
                "word_count",
                    "char_count",
                        "impressions_90d",
                            "clicks_90d",
                                "pageviews_90d",
                                    "sessions_90d",
                                        "users_90d",
                                            "engaged_sessions_90d",
                                                "ai_sessions_90d",
                                                    "scroll_events_90d",
                                                        "days_with_impressions",
                                                            "days_with_sessions",
                                                                "impressions_last_30d",
                                                                    "clicks_last_30d",
                                                                        "sessions_last_30d",
                                                                            "impressions_prev_30d",
                                                                                "clicks_prev_30d",
                                                                                    "sessions_prev_30d",
                                                                                        "content_age_days",
                                                                                            "days_since_last_update",
                                                                                                "ctr",
                                                                                                    "avg_position",
                                                                                                        "engagement_rate",
                                                                                                            "scroll_rate",
                                                                                                                "ai_traffic_pct"
                                                                                                                ]

                                                                                                                # Target
target_col = "is_declining_label"

                                                                                                                # Columns deliberately excluded to prevent target leakage
excluded_cols = [
                                                                                                                    "trend_direction",
                                                                                                                        "trend_pct"
                                                                                                                        ]

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Number of features:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExcluded leakage-prone columns:")
for col in excluded_cols:
                                                                                                                            print("-", col)

                                                                                                                            print("\nMissing values in features:", X.isna().sum().sum())

Number of features: 28
X shape: (30000, 28)
y shape: (30000,)

Excluded leakage-prone columns:
- trend_direction

Missing values in features: 22927
- trend_pct

Missing values in features: 22927


In [13]:
# Inspect feature data types and missing values

print("Feature data types:")
print(X.dtypes.value_counts())

print("\nTop features by missing-value count:")
missing_summary = (
    X.isna()
        .sum()
            .sort_values(ascending=False)
            )

print(missing_summary.head(10))

print("\nTarget missing values:", y.isna().sum())

Feature data types:
int64      18
float64    10
Name: count, dtype: int64

Top features by missing-value count:
word_count         7699
char_count         7699
competition        2468
search_volume      2468
cpc                2468
scroll_rate         125
clicks_90d            0
impressions_90d       0
sessions_90d          0
users_90d             0
dtype: int64

Target missing values: 0


In [14]:
from sklearn.model_selection import GroupShuffleSplit

# Grouped train/test split
# This prevents the same client from appearing in both sets.
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
        test_size=0.20,
            random_state=42
            )

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())
print("\nClient overlap:",
                  len(set(groups_train) & set(groups_test)))

print("\nTrain declining rate:", round(y_train.mean(), 4))
print("Test declining rate:", round(y_test.mean(), 4))

Train shape: (23837, 28)
Test shape: (6163, 28)

Train clients: 25
Test clients: 7

Client overlap: 0

Train declining rate: 0.5501
Test declining rate: 0.511


In [17]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
        precision_score,
            recall_score,
                f1_score,
                    roc_auc_score
                    )

                    # Majority-class baseline
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_prob = baseline.predict_proba(X_test)[:, 1]

baseline_metrics = {
                        "Accuracy": accuracy_score(y_test, baseline_pred),
                            "Precision": precision_score(y_test, baseline_pred, zero_division=0),
                                "Recall": recall_score(y_test, baseline_pred, zero_division=0),
                                    "F1": f1_score(y_test, baseline_pred, zero_division=0),
                                        "ROC-AUC": roc_auc_score(y_test, baseline_prob)
                                        }

print("Majority Baseline Results:")
for metric, value in baseline_metrics.items():
                                            print(f"{metric}: {value:.4f}")

Majority Baseline Results:
Accuracy: 0.5110
Precision: 0.5110
Recall: 1.0000
F1: 0.6763
ROC-AUC: 0.5000


In [20]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Build the modeling pipeline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                    max_iter=1000,
                            random_state=42
                                ))
                                ])

                                # Train only on the training set

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


## 1. Question

*The research question and the decision it supports.*

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
